# Sequence Models: RNNs and LSTMs

## Overview

Sequence models process ordered data where context from earlier timesteps informs current predictions. They are the standard architecture for time series, sensor streams, and text — anywhere temporal or sequential order matters.

**Architecture comparison:**

| Model | Memory | Vanishing gradient | Speed | When to use |
|---|---|---|---|---|
| Simple RNN | Short-term | Severe | Fast | Very short sequences |
| LSTM | Long + short-term | Mitigated | Medium | General sequences |
| GRU | Balanced | Mitigated | Faster than LSTM | Good default |
| Temporal CNN | Fixed window | None | Fastest | Long sequences, parallelisable |
| Transformer | Full attention | None | Scalable | Long-range dependencies |

**LSTM gate mechanics:**
- Forget gate: how much of previous cell state to retain
- Input gate: how much of new input to write to cell state
- Output gate: how much of cell state to expose as hidden state

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from copy import deepcopy

torch.manual_seed(42)
rng = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Simulate multivariate water quality time series
# Task: predict next-step nitrate from rolling window of [nitrate, phosphorus, flow]
n_series   = 500
seq_len    = 30     # input window
pred_ahead = 1      # predict 1 step ahead

# Generate AR(1) process with seasonal component
t = np.arange(n_series + seq_len + pred_ahead)
nitrate_ts   = (3 + 1.5*np.sin(2*np.pi*t/52) +
                np.cumsum(rng.normal(0, 0.1, len(t)))).clip(0.5)
phosphorus_ts = 0.6*nitrate_ts + rng.normal(0, 0.3, len(t))
flow_ts      = (10 + 5*np.cos(2*np.pi*t/52) +
                rng.normal(0, 1.5, len(t))).clip(1)
data_3d = np.column_stack([nitrate_ts, phosphorus_ts, flow_ts]).astype(np.float32)

# Normalise
mu, sd = data_3d[:n_series].mean(0), data_3d[:n_series].std(0) + 1e-8
data_n = (data_3d - mu) / sd

# Build sliding window dataset
X_seqs, y_seqs = [], []
for i in range(n_series):
    X_seqs.append(data_n[i:i+seq_len])
    y_seqs.append(data_n[i+seq_len, 0])   # predict nitrate
X_t = torch.from_numpy(np.array(X_seqs))   # (n, seq_len, 3)
y_t = torch.from_numpy(np.array(y_seqs)).unsqueeze(1)
split = int(0.8*n_series)
X_tr, X_va = X_t[:split], X_t[split:]
y_tr, y_va = y_t[:split], y_t[split:]
train_dl = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)
print(f"X shape: {X_t.shape}  (n_samples, seq_len, n_features)")
print(f"y shape: {y_t.shape}  (n_samples, 1)")

---
## LSTM and GRU Models

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers,
                            dropout=dropout if num_layers > 1 else 0,
                            batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])   # take last timestep

class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers=num_layers,
                           dropout=dropout if num_layers > 1 else 0,
                           batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(out[:, -1, :])

def train_seq(model, epochs=80, lr=1e-3):
    model = model.to(device)
    opt   = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    crit  = nn.MSELoss()
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_val, best_state = float('inf'), None
    va_hist = []
    for ep in range(epochs):
        model.train()
        for Xb, yb in train_dl:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad(); crit(model(Xb), yb).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vl = crit(model(X_va.to(device)), y_va.to(device)).item()
        va_hist.append(vl)
        sched.step()
        if vl < best_val:
            best_val = vl; best_state = deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, va_hist

lstm_model, lstm_hist = train_seq(LSTMModel(input_size=3))
gru_model,  gru_hist  = train_seq(GRUModel(input_size=3))
print(f"LSTM best val RMSE: {np.sqrt(min(lstm_hist)):.4f}")
print(f"GRU  best val RMSE: {np.sqrt(min(gru_hist)):.4f}")

---
## Predictions and Error Analysis

In [ ]:
lstm_model.eval(); gru_model.eval()
with torch.no_grad():
    y_pred_lstm = lstm_model(X_va.to(device)).cpu().numpy().ravel()
    y_pred_gru  = gru_model(X_va.to(device)).cpu().numpy().ravel()
y_true = y_va.numpy().ravel()

# Rescale back to original units (nitrate only, dim 0)
y_true_r      = y_true * sd[0] + mu[0]
y_pred_lstm_r = y_pred_lstm * sd[0] + mu[0]
y_pred_gru_r  = y_pred_gru  * sd[0] + mu[0]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(lstm_hist, label='LSTM', color='steelblue')
axes[0].plot(gru_hist,  label='GRU',  color='#e74c3c')
axes[0].set_title('Val Loss (MSE)'); axes[0].legend()
n_show = 80
axes[1].plot(y_true_r[:n_show], label='True',  color='black', lw=1.5)
axes[1].plot(y_pred_lstm_r[:n_show], label='LSTM', color='steelblue', lw=1.5, alpha=0.8)
axes[1].plot(y_pred_gru_r[:n_show],  label='GRU',  color='#e74c3c', lw=1.5, alpha=0.8)
axes[1].set_title('Predictions vs True (first 80 val steps)')
axes[1].legend()
resid = y_true_r - y_pred_lstm_r
axes[2].scatter(y_pred_lstm_r, resid, s=15, alpha=0.5, color='steelblue')
axes[2].axhline(0, color='black', lw=1)
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Residual')
axes[2].set_title('LSTM Residuals')
plt.tight_layout(); plt.show()
rmse_lstm = np.sqrt(np.mean((y_true_r-y_pred_lstm_r)**2))
rmse_gru  = np.sqrt(np.mean((y_true_r-y_pred_gru_r)**2))
print(f"LSTM RMSE (original units): {rmse_lstm:.4f}")
print(f"GRU  RMSE (original units): {rmse_gru:.4f}")

---
## Multi-Step Ahead Forecasting

In [ ]:
def forecast_multistep(model, seed_seq, n_steps, device):
    model.eval()
    seq = seed_seq.clone()   # (seq_len, n_features)
    preds = []
    with torch.no_grad():
        for _ in range(n_steps):
            inp  = seq.unsqueeze(0).to(device)   # (1, seq_len, n_features)
            pred = model(inp).item()              # normalised nitrate
            preds.append(pred)
            # Roll window: append prediction, drop oldest
            new_step = seq[-1].clone()
            new_step[0] = pred   # update nitrate dim
            seq = torch.cat([seq[1:], new_step.unsqueeze(0)], dim=0)
    return np.array(preds)

n_ahead  = 20
seed     = X_va[0]   # first val window as seed
forecast = forecast_multistep(lstm_model, seed, n_ahead, device)
# Rescale
forecast_r = forecast * sd[0] + mu[0]
true_future = data_3d[split+seq_len:split+seq_len+n_ahead, 0]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(n_ahead), true_future,  'o-', label='True',     color='black', lw=2)
ax.plot(range(n_ahead), forecast_r,   's--', label='Forecast', color='steelblue', lw=2)
ax.set_xlabel('Steps ahead'); ax.set_ylabel('Nitrate (mg/L)')
ax.set_title('Multi-Step Ahead Forecast (LSTM)')
ax.legend(); plt.tight_layout(); plt.show()
print("Note: error accumulates in multi-step forecasting because predicted")
print("values are fed back as inputs -- uncertainty grows with horizon.")

In [ ]:
# Attention mechanism: simple additive attention over LSTM outputs
class LSTMWithAttention(nn.Module):
    def __init__(self, input_size, hidden_size=64):
        super().__init__()
        self.lstm    = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.attn    = nn.Linear(hidden_size, 1)
        self.head    = nn.Sequential(nn.Linear(hidden_size, 32),
                                      nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        out, _ = self.lstm(x)                       # (B, T, H)
        weights = torch.softmax(self.attn(out), dim=1)  # (B, T, 1)
        context = (weights * out).sum(dim=1)         # (B, H)
        return self.head(context)

attn_model, attn_hist = train_seq(LSTMWithAttention(input_size=3))
attn_model.eval()
with torch.no_grad():
    y_pred_attn = attn_model(X_va.to(device)).cpu().numpy().ravel()
rmse_attn = np.sqrt(np.mean(((y_pred_attn - y_va.numpy().ravel()) * sd[0])**2))
print(f"Attention LSTM RMSE: {rmse_attn:.4f}")
# Visualise attention weights for one example
attn_model.eval()
with torch.no_grad():
    sample_seq = X_va[:1].to(device)
    out_seq, _ = attn_model.lstm(sample_seq)
    attn_w     = torch.softmax(attn_model.attn(out_seq), dim=1)
    attn_w_np  = attn_w[0,:,0].cpu().numpy()
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(range(seq_len), attn_w_np, color='steelblue', alpha=0.8)
ax.set_xlabel('Timestep'); ax.set_ylabel('Attention weight')
ax.set_title('Attention Weights: Which Timesteps Matter Most?')
plt.tight_layout(); plt.show()

---

## Common Pitfalls

**1. Using `batch_first=False` (the default) and passing `(batch, seq, features)` shaped input**  
PyTorch LSTM and GRU default to `batch_first=False`, expecting input of shape `(seq_len, batch, features)`. If your data is `(batch, seq_len, features)` — the more natural format — always set `batch_first=True`. A shape mismatch produces silent wrong results or a cryptic dimension error.

**2. Taking the last hidden state without checking if sequences are padded**  
For variable-length sequences padded to equal length, the last timestep output (`out[:, -1, :]`) corresponds to the padding, not the end of the actual sequence. Use `pack_padded_sequence` and `pad_packed_sequence` with `enforce_sorted=False` to correctly handle variable-length inputs.

**3. Using RNN/LSTM for very long sequences without attention**  
LSTMs struggle to maintain relevant information across hundreds of timesteps. For sequences longer than ~100 steps, consider temporal CNNs, transformer encoders, or adding an attention mechanism over LSTM outputs to give the model direct access to relevant historical timesteps.

**4. Forecasting multiple steps ahead by feeding predictions back without quantifying uncertainty**  
Multi-step forecasting compounds errors — each predicted value is fed back as input, and uncertainty grows exponentially with horizon. Always plot forecast intervals and report RMSE at each forecast horizon separately to communicate how rapidly prediction quality degrades.

**5. Not shuffling training sequences when using a DataLoader**  
LSTMs do not require sequences to be fed in temporal order during training (they process each window internally). Shuffling batches prevents the model from learning spurious patterns from the ordering of training examples. Always use `shuffle=True` in the DataLoader.
---
*python_methods_library - Samantha McGarrigle*